In [1]:
# conda activate chronocell

import os
import sys
import pandas as pd

os.chdir("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/janssens_2025_preprint")

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")
sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/janssens_2025_preprint/code")
# sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/simulations/code")

import Chronocell
from reconstruct_RNA_history import *
# from protein_from_RNA import *

In [2]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.5_208_genes_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [3]:
Y = traj.X
Q = traj.Q[:, 0, :] 
tau = traj.tau # State transition times (global)
t = traj.t
theta = traj.theta
topo = traj.topo
state_grid = np.searchsorted(tau, t, side="left") - 1
state_grid[0] = 0 

# theta_ = theta.copy()
# a0 = theta_[:, 0] # Starting RNA abundance 
# a = theta_[:, 1:len(topo.flatten())] 
# beta = theta_[:, -2] # Splicing rate
# alpha = a * beta[:, None] # These values are divided by splicing rate; removing this factor now
# gamma = theta_[:, -1] # Degradation rate

## Subset to genes with protein measurements

### How many counts are in cells for genes of interest?

In [4]:
genes = pd.read_csv("eLNPs_var>1.5_208_genes.csv")
shared_genes = pd.read_csv("RNA_vs_ADT_corr_meanExpr.csv")

In [5]:
gene_idx = genes['Gene_symbol'].isin(shared_genes['Gene']).tolist()

In [6]:
shared_genes_mean_expr_U = Y[:, gene_idx, 0].max(axis=0)
shared_genes_mean_expr_S = Y[:, gene_idx, 1].max(axis=0)

In [7]:
np.sort(shared_genes_mean_expr_U)

array([  1.,   1.,   1.,   1.,   2.,   2.,   3.,   3.,   3.,   3.,   3.,
         3.,   3.,   4.,   4.,   4.,   4.,   5.,   6.,   6.,   6.,   6.,
         6.,   7.,   7.,   7.,   9.,  10.,  11.,  11.,  12.,  14.,  14.,
        16.,  16.,  16.,  19.,  19.,  22.,  22.,  24.,  25.,  27.,  27.,
        38.,  42.,  46.,  54.,  56.,  60., 115.])

In [8]:
np.sort(shared_genes_mean_expr_S)

array([  3.,   4.,   4.,   4.,   5.,   5.,   5.,   5.,   6.,   6.,   7.,
         7.,   7.,   7.,   7.,   8.,   8.,   8.,   8.,   8.,   8.,   9.,
        10.,  10.,  10.,  11.,  11.,  11.,  12.,  12.,  12.,  13.,  13.,
        14.,  16.,  16.,  16.,  18.,  20.,  21.,  25.,  27.,  27.,  28.,
        35.,  35.,  41.,  43.,  54.,  99., 105.])

In [9]:
Y = Y[:, gene_idx, :]

In [10]:
theta = theta[gene_idx, :]
theta_ = theta.copy()
a0 = theta_[:, 0] # Starting RNA abundance 
a = theta_[:, 1:len(topo.flatten())] 
beta = theta_[:, -2] # Splicing rate
alpha = a * beta[:, None] # These values are divided by splicing rate; removing this factor now
gamma = theta_[:, -1] # Degradation rate

## Reconstruct RNA

In [16]:
def max_RNAs(Y, gene_idx, frac=0.1):
    U_max = np.max(Y[:, gene_idx, 0])
    S_max = np.max(Y[:, gene_idx, 1])
    U_max = (U_max + np.ceil(U_max * frac)).astype("int")
    S_max = (S_max + np.ceil(S_max * frac)).astype("int")
    return U_max, S_max

### Dense implementation

In [ ]:
X_bw_per_gene = []
states_per_gene = []

for gene_idx in range(0, Y.shape[0]):
    print("Starting gene", gene_idx)

    # Set max # of RNAs based on observed values for a given gene
    U_max, S_max = max_RNAs(Y, gene_idx)
    states, index_for = enumerate_states(U_max, S_max)

    print("U_max, S_max:", U_max, S_max)

    beta_j = beta[gene_idx]
    gamma_j = gamma[gene_idx] 
    alpha_j = alpha[gene_idx, :]
    Y_j = Y[:, gene_idx, :]
    
    # Make generator matrix (per transcription rate)
    A = []
    for i in range(0, alpha.shape[1]):
        rxns = define_reactions(alpha_j[i], beta_j, gamma_j)
        A1 = create_transition_matrix(rxns, states, index_for, U_max, S_max)
        A.append(A1)
    
    # Calculate forward probability distribution (needed for reverse generator)
    alpha0 = a0[gene_idx] * beta_j
    pi = stationary_from_params(alpha0, beta_j, gamma_j, states) 
    X_fwd = forward_distribution(A, pi, states, t, tau, state_grid)

    # Calculate backward probability distribution (per cell)
    X_bw = backward_distribution(Y_j, Q, states, index_for, t, tau, state_grid)
        
    X_bw_per_gene.append(X_bw)
    states_per_gene.append(states)
    
    get_expm.cache_clear()
    get_A_rev.cache_clear()
    get_expm_rev.cache_clear()
    

Starting gene 0
U_max, S_max: 5 14
Starting gene 1
U_max, S_max: 16 9
Starting gene 2
U_max, S_max: 4 8
Starting gene 3
U_max, S_max: 7 6
Starting gene 4
U_max, S_max: 4 18
Starting gene 5
U_max, S_max: 4 48


KeyboardInterrupt: 

In [12]:
get_expm.cache_clear()
get_A_rev.cache_clear()
get_expm_rev.cache_clear()

### Sparse implementation

In [ ]:
X_bw_per_gene = []
states_per_gene = []
    
for gene_idx in range(0, Y.shape[0]):
    print("Starting gene", gene_idx)

    # Set max # of RNAs based on observed values for a given gene
    U_max = np.max(Y[:, gene_idx, 0])
    S_max = np.max(Y[:, gene_idx, 1])
    U_max = (U_max + np.ceil(U_max * .1)).astype("int")
    S_max = (S_max + np.ceil(S_max * .1)).astype("int")
    
    # if U_max < X, skip
    
    states, index_for = enumerate_states(U_max, S_max)
    
    
    print("U_max, S_max:", U_max, S_max)
    
    beta_j = beta[gene_idx]
    gamma_j = gamma[gene_idx] 
    alpha_j = alpha[gene_idx, :]
    
    # Make generator matrix (per transcription rate)
    A = []
    for i in range(0, alpha.shape[1]):
        rxns = define_reactions(alpha_j[i], beta_j, gamma_j)
        A1 = create_transition_matrix_sparse(rxns, states, index_for, U_max, S_max)
        A.append(A1)
     
    # Calculate forward probability distribution (needed for reverse generator)
    alpha0 = a0[gene_idx] * beta_j
    pi = stationary_from_params(alpha0, beta_j, gamma_j, states)
    X_fwd = forward_distribution_blocked_sparse(A, pi, states, t, tau, state_grid)
    
    # Calculate backward probability distribution (per cell)
    X_bw_per_cell = [] 
    for cell_idx in range(0, Q.shape[0]):
        X_bw = backward_distribution_sparse(Y[:, gene_idx, :], Q, states, index_for, t, tau, state_grid)
        X_bw_per_cell.append(X_bw)
        
    X_bw_per_gene.append(X_bw_per_cell)
    states_per_gene.append(states)
    
    get_A_rev_sparse.cache_clear()

Starting gene 0
U_max, S_max: 5 14


KeyboardInterrupt: 

In [ ]:
# Y_observed, Y, theta, rd, true_t, true_l = simulate_RNA(topo, tau, theta[0, :][None, :], n=20000, random_seed=666)